# 05 - Evaluation, error analysis and Grad-CAM

Evaluate the final model (`models/medical_model.keras`) on the untouched test set, then look at *where* it fails and *where it looks*.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make the project's `src` package importable from inside notebooks/
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), Path.cwd())
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src import config
from src.utils import find_dataset_root, count_images_per_class

# Point this at data/dummy to try the notebook without the real dataset
DATA_DIR = config.DATA_DIR
ROOT_DATA = find_dataset_root(DATA_DIR)
print("Dataset root:", ROOT_DATA)

import pandas as pd
from IPython.display import Image as IPImage, display
from src import evaluate
from src.predict import load_model_and_config, predict_image

## Metrics on the test set

Two operating points are reported: the default threshold 0.5, and the threshold chosen on the **validation** set to reach the recall target. For a screening-style task, recall (sensitivity) matters most because a missed case costs more than a false alarm.

In [ ]:
results = evaluate.main(["--data-dir", str(DATA_DIR)])
pd.DataFrame(results).T.round(4)

In [ ]:
for name in ("confusion_matrix.png", "roc_curve.png"):
    display(IPImage(filename=str(config.OUTPUTS_DIR / name)))

## Error analysis

Top row: pneumonia the model missed (false negatives). Bottom row: normal X-rays it flagged (false positives). Look for patterns - image quality, devices, borders, text markers.

In [ ]:
display(IPImage(filename=str(config.OUTPUTS_DIR / "error_analysis.png")))

## Grad-CAM

A heatmap of the image regions that most increased the score of the predicted class. Sanity check: for correct pneumonia predictions the heat should sit on the lungs, not on text labels, borders or medical devices.

In [ ]:
model, cfg = load_model_and_config()
threshold = cfg["threshold"]

preds = pd.read_csv(config.OUTPUTS_DIR / "test_predictions.csv")
samples = pd.concat([
    preds[(preds.y_true == 1) & (preds.y_pred == 1)].head(2),   # true positives
    preds[(preds.y_true == 0) & (preds.y_pred == 0)].head(2),   # true negatives
    preds[(preds.y_true == 1) & (preds.y_pred == 0)].head(2),   # false negatives
    preds[(preds.y_true == 0) & (preds.y_pred == 1)].head(2),   # false positives
])

fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
for ax, (_, row) in zip(axes.ravel(), samples.iterrows()):
    r = predict_image(model, row.path, cfg["img_size"], threshold, with_gradcam=True)
    ax.imshow(r["overlay"]); ax.axis("off")
    actual = config.CLASS_NAMES[int(row.y_true)]
    ax.set_title(f"actual {actual}\npredicted {r['label']} ({r['probability']:.0%})", fontsize=10)
plt.tight_layout(); plt.show()

## Write-up for your README

Fill in the results table in the README with the numbers above, and describe 2-3 failure patterns you found. That analysis is what makes the project stand out in interviews.